# 03 — BQ1: Dove e quando abbandonano gli studenti?

> **Notebook 03 di 7** | Learning Retention Analytics  
> Analisi della prima business question: pattern temporali di abbandono degli studenti nei vari corsi.

## Scopo e ambito

Questo notebook risponde a **BQ1: Dove e quando abbandonano gli studenti?** — la prima delle cinque business question che guidano il progetto.

**Cosa copre questo notebook:**
- Dimensione del problema dropout: quanti studenti si ritirano e da quali corsi
- Curve cumulative di dropout — analisi survival-style che mostra quando gli studenti se ne vanno
- Rilevamento dei cliff — identificazione dei momenti critici di abbandono di massa
- Timeline normalizzata — confronto dei tempi di dropout tra corsi di durata diversa
- Ritiri pre-corso — studenti che se ne vanno prima ancora che il corso inizi
- Segmentazione demografica dei tempi di dropout
- Caratteristiche di progettazione del corso e relazione con i pattern di dropout

**Cosa viene dopo:**
- **Notebook 04** (`04_bq2_early_signals.ipynb`): quali segnali comportamentali precoci predicono il dropout? (BQ2)

**Collegamento con le altre business question:**  
BQ1 stabilisce *quando* gli studenti se ne vanno. BQ2 (Notebook 04) identificherà i segnali comportamentali che *predicono* l'abbandono. Insieme, definiscono sia la finestra di intervento sia i trigger per i sistemi di early warning.

> **Trasferibilità metodologica:** L'analisi survival-style del dropout, il rilevamento dei cliff e il confronto su timeline normalizzata utilizzati qui sono direttamente applicabili all'analisi del churn SaaS (quando cancellano gli abbonati?), al decadimento delle coorti di abbonamento (quali coorti trattengono meglio?) e al calo di engagement nelle app (quando gli utenti smettono di tornare?). Il dominio è l'istruzione; il framework analitico è la retention analytics.

## Indice

1. [Configurazione ambiente](#1.-Environment-Setup)
2. [Panoramica dropout — Dimensione e tasso](#2.-Dropout-Overview-—-Scale-and-Rate)
3. [Curve cumulative di dropout](#3.-Cumulative-Dropout-Curves)
4. [Identificazione dei cliff di dropout](#4.-Identifying-Dropout-Cliffs)
5. [Timeline normalizzata — Percentuale di avanzamento nel corso](#5.-Normalized-Timeline-—-Percentage-Through-Course)
6. [Ritiri pre-corso](#6.-Pre-Course-Withdrawals)
7. [Tempi di dropout per dati demografici](#7.-Dropout-Timing-by-Demographics)
8. [Progettazione del corso e pattern di dropout](#8.-Course-Design-and-Dropout-Patterns)
9. [Conclusioni chiave e prossimi passi](#9.-Key-Takeaways-and-Next-Steps)

---

**Prerequisiti:**
- La pipeline ETL deve essere stata eseguita: `python -m run_pipeline`
- Il database DuckDB in `data/db/oulad.duckdb` deve contenere tutte e 5 le viste analitiche

**Dataset:** Open University Learning Analytics Dataset (OULAD) — ~32K studenti, 7 corsi, clickstream comportamentale completo. Licenza: CC-BY 4.0.

## 1. Configurazione ambiente

Configuriamo gli import, i parametri di visualizzazione e le funzioni helper riutilizzabili.

**Note tecniche per il lettore:**
- I notebook si trovano in `notebooks/` ma i moduli del progetto sono in `src/` nella root. Aggiungiamo la root del progetto a `sys.path` affinché `from src.config import ...` funzioni.
- Tutte le query al database passano attraverso `src.db.connection.execute_query()` — il livello di astrazione DB del progetto (ADR-003).
- La query SQL principale di BQ1 risiede in `sql/queries/q_bq1_dropout_curves.sql` e viene caricata a runtime dal disco. Query aggiuntive in questo notebook possono essere definite inline, ma vengono comunque eseguite attraverso lo stesso livello di astrazione DB (ADR-003).
- Le figure vengono salvate in `reports/figures/` a 150 DPI. Poiché `nbstripout` rimuove gli output dei notebook prima del commit, i PNG salvati sono il record visivo persistente.

In [ ]:
# --- Path setup ---
# Notebooks live in notebooks/ but project modules are at the project root.
# We search upward for pyproject.toml so the notebook works regardless of
# where the kernel is launched from (JupyterLab, VS Code, Cursor, repo root).
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from start until we find pyproject.toml (the repo root marker)."""
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate project root: no pyproject.toml found in '{start}' "
        "or any parent directory. Run the notebook from within the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Standard library ---
import logging
import warnings

# --- Third-party ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Project modules ---
from src.config import FIGURES_DIR, QUERIES_DIR
from src.db.connection import execute_query

# --- Configuration ---
# Suppress noisy warnings in notebook output; errors still surface
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

# --- Visualization defaults ---
# Consistent style across all project notebooks
sns.set_theme(style='whitegrid', font_scale=1.1)

# Semantic color palette: one color per outcome category
PALETTE_OUTCOME = {
    'Pass': '#4C72B0',        # blue — neutral positive
    'Distinction': '#55A868', # green — strong positive
    'Fail': '#C44E52',        # red — negative
    'Withdrawn': '#8172B3',   # purple — departed (distinct from failed)
}
# Outcome label constants — single source of truth for binary labels
LABEL_COMPLETED = 'Completed'
LABEL_NOT_COMPLETED = 'Not completed'
PALETTE_BINARY = {1: '#55A868', 0: '#C44E52'}
LABEL_BINARY = {1: LABEL_COMPLETED, 0: LABEL_NOT_COMPLETED}
PALETTE_BINARY_LABELS = {LABEL_COMPLETED: '#55A868', LABEL_NOT_COMPLETED: '#C44E52'}
PALETTE_SEQUENTIAL = 'YlOrRd'

# Course-level palette: one color per module for consistent identification
# across all dropout curve and comparison charts. The 7 OULAD modules use
# tab10 for maximum visual distinction between course lines.
_MODULE_ORDER = ['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG']
_TAB10 = plt.cm.tab10.colors
PALETTE_COURSE = {m: _TAB10[i] for i, m in enumerate(_MODULE_ORDER)}

# Shared axis labels — avoids cross-cell string literal duplication
LABEL_DROPOUT_DAY = 'Dropout day (relative to course start)'
LABEL_CUMULATIVE_DROPOUT = 'Cumulative dropout rate (%)'
LABEL_COMPLETION_RATE = 'Completion rate (%)'
LABEL_MEDIAN_DROPOUT_DAY = 'Median dropout day'

FIG_DPI = 150
FIG_SIZE = (10, 6)
FIG_SIZE_WIDE = (16, 5)

# Ensure figures output directory exists
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save figure to reports/figures/ with consistent settings."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')


# --- Load BQ1 query from SQL file ---
# The primary query lives in sql/queries/ as a standalone file (ADR-003).
# Loading it once here avoids re-reading the file in multiple cells.
_bq1_sql_path = QUERIES_DIR / 'q_bq1_dropout_curves.sql'
BQ1_SQL = _bq1_sql_path.read_text(encoding='utf-8')
print(f'Loaded BQ1 query from: {_bq1_sql_path.name} ({len(BQ1_SQL):,} chars)')

# --- Prerequisite check ---
# Verify the database is populated before proceeding.
try:
    _check_student = execute_query('SELECT COUNT(*) AS n FROM v_student_enriched')
    _check_dropout = execute_query('SELECT COUNT(*) AS n FROM v_dropout_timing')
    _n_student = _check_student['n'].iloc[0]
    _n_dropout = _check_dropout['n'].iloc[0]
    if _n_student == 0 or _n_dropout == 0:
        raise RuntimeError('One or more views are empty')
    print('Database OK')
    print(f'  v_student_enriched:  {_n_student:>12,} rows')
    print(f'  v_dropout_timing:    {_n_dropout:>12,} rows')
except Exception as exc:
    raise RuntimeError(
        'Cannot query analytical views. '
        "Run 'python -m run_pipeline' first to populate the database."
    ) from exc

## 2. Panoramica dropout — Dimensione e tasso

Prima di esaminare *quando* gli studenti abbandonano, stabiliamo **quanti** abbandonano e da quali corsi. Questa sezione fornisce i numeri di base che contestualizzano tutte le analisi successive.

**Distinzione chiave:** Nel dataset OULAD, "withdrawn" significa che lo studente si è esplicitamente disiscritto dal corso (`date_unregistration IS NOT NULL`). Gli studenti che sono rimasti iscritti ma non hanno superato l'esame sono classificati come "Fail", non "Withdrawn". Questo notebook si concentra sugli **abbandoni espliciti** — gli studenti che hanno attivamente scelto di andarsene.

In [ ]:
# --- Dropout headline numbers ---
# Count explicit withdrawals (withdrew_explicit = 1) vs total enrollments
# per module to establish the scale of the dropout problem.
df_overview = execute_query('''
    SELECT
        code_module,
        COUNT(*) AS n_enrolled,
        SUM(withdrew_explicit) AS n_withdrew,
        ROUND(100.0 * SUM(withdrew_explicit) / COUNT(*), 1) AS withdrawal_rate_pct
    FROM v_student_enriched
    GROUP BY code_module
    ORDER BY withdrawal_rate_pct DESC
''')

# Overall numbers
total_enrolled = df_overview['n_enrolled'].sum()
total_withdrew = df_overview['n_withdrew'].sum()
overall_rate = 100.0 * total_withdrew / total_enrolled

print('=== Dropout Overview ===\n')
print(f'  Total enrollments:     {total_enrolled:>8,}')
print(f'  Explicit withdrawals:  {total_withdrew:>8,} ({overall_rate:.1f}%)')
print(f'  Stayed enrolled:       {total_enrolled - total_withdrew:>8,} ({100 - overall_rate:.1f}%)')
print('\n=== Withdrawal Rate by Module ===\n')
print(df_overview.to_string(index=False))

# --- Horizontal bar chart: withdrawal count + rate per module ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

modules = df_overview['code_module']
colors = [PALETTE_COURSE[m] for m in modules]

# Left: absolute withdrawal count
ax1.barh(modules, df_overview['n_withdrew'], color=colors, edgecolor='white')
for i, (_, row) in enumerate(df_overview.iterrows()):
    ax1.text(
        row['n_withdrew'] + ax1.get_xlim()[1] * 0.01, i,
        f"{int(row['n_withdrew']):,}",
        va='center', fontsize=9, color='#333333',
    )
ax1.set_xlabel('Number of withdrawals')
ax1.set_title('Withdrawal Count by Module')
ax1.invert_yaxis()
sns.despine(ax=ax1)

# Right: withdrawal rate percentage
ax2.barh(modules, df_overview['withdrawal_rate_pct'], color=colors, edgecolor='white')
for i, (_, row) in enumerate(df_overview.iterrows()):
    ax2.text(
        row['withdrawal_rate_pct'] + 0.5, i,
        f"{row['withdrawal_rate_pct']:.1f}%",
        va='center', fontsize=9, color='#333333',
    )
# Reference line: overall withdrawal rate across all modules
ax2.axvline(x=overall_rate, color='gray', linestyle='--', linewidth=1,
            label=f'Overall: {overall_rate:.1f}%')
ax2.set_xlabel('Withdrawal rate (%)')
ax2.set_title('Withdrawal Rate by Module')
ax2.legend(loc='lower right')
ax2.invert_yaxis()
sns.despine(ax=ax2)

fig.suptitle('Dropout Overview by Module', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '03_dropout_overview_by_course')
plt.show()

> **Risultato chiave:** Circa un terzo di tutte le iscrizioni si conclude con un ritiro esplicito. Il tasso di ritiro varia tra i moduli — alcuni corsi perdono una quota significativamente maggiore di studenti rispetto ad altri.
>
> Questa variazione è importante: BQ4 (Notebook 06) indagherà se le caratteristiche di progettazione del corso (densità delle valutazioni, numero di risorse, durata) spiegano queste differenze. Per ora, notiamo che il problema è sostanziale e dipendente dal modulo.

## 3. Curve cumulative di dropout

Questa è la **visualizzazione centrale di BQ1** — un'analisi survival-style che mostra come il dropout si accumula nel tempo.

La query `q_bq1_dropout_curves.sql` calcola conteggi e percentuali cumulative di dropout per corso-presentazione utilizzando window function (`SUM OVER ... ROWS BETWEEN UNBOUNDED PRECEDING`). Ogni punto sulla curva rappresenta la percentuale della coorte originale che si è ritirata entro quel giorno.

**Perché grafici a gradini?** Il dropout è un evento discreto — gli studenti si ritirano in giorni specifici, non in modo continuo. Un grafico a gradini (anziché una linea smussata) preserva questa realtà e rende visibili gli eventi cliff (picchi improvvisi).

In [ ]:
# --- Execute BQ1 query ---
# The SQL file was loaded in the setup cell. Each row represents a day
# when at least one dropout occurred, with running totals and percentages.
df_curves = execute_query(BQ1_SQL)

print(f'BQ1 result: {len(df_curves):,} rows')
print(f'Courses: {df_curves["code_module"].nunique()} modules, '
      f'{df_curves[["code_module", "code_presentation"]].drop_duplicates().shape[0]} presentations')

# --- Overlaid cumulative dropout curves ---
# One line per course-presentation, colored by module. This shows the
# full landscape: how different cohorts decay over time.
fig, ax = plt.subplots(figsize=(12, 7))

for (module, pres), group in df_curves.groupby(['code_module', 'code_presentation']):
    ax.step(
        group['dropout_day'], group['cumulative_dropout_rate_pct'],
        where='post', color=PALETTE_COURSE[module], alpha=0.7,
        linewidth=1.5, label=None,
    )

# Add one legend entry per module (not per presentation)
for module in _MODULE_ORDER:
    if module in df_curves['code_module'].values:
        ax.plot([], [], color=PALETTE_COURSE[module], linewidth=2, label=module)

ax.set_xlabel(LABEL_DROPOUT_DAY)
ax.set_ylabel(LABEL_CUMULATIVE_DROPOUT)
ax.set_title('Cumulative Dropout Curves — All Course-Presentations')
ax.legend(title='Module', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_ylim(0, None)
sns.despine()
fig.tight_layout()
save_fig(fig, '03_dropout_curves_overlaid')
plt.show()

In [ ]:
# --- Faceted small multiples: one panel per module ---
# Separating modules into individual panels avoids the visual clutter of
# the overlaid view and lets the reader focus on each course's trajectory.
modules_present = sorted(df_curves['code_module'].unique())
n_modules = len(modules_present)
n_cols = 4
n_rows = (n_modules + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows), sharey=True)
axes_flat = axes.flatten()

for i, module in enumerate(modules_present):
    ax = axes_flat[i]
    module_data = df_curves[df_curves['code_module'] == module]

    for pres, group in module_data.groupby('code_presentation'):
        ax.step(
            group['dropout_day'], group['cumulative_dropout_rate_pct'],
            where='post', color=PALETTE_COURSE[module], alpha=0.7,
            linewidth=1.5, label=pres,
        )

    ax.set_title(module, fontsize=12, fontweight='bold')
    ax.set_xlabel(LABEL_DROPOUT_DAY)
    if i % n_cols == 0:
        ax.set_ylabel(LABEL_CUMULATIVE_DROPOUT)
    ax.legend(fontsize=7, loc='lower right')
    ax.set_ylim(0, None)
    sns.despine(ax=ax)

# Hide unused subplots
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle('Cumulative Dropout Curves by Module', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '03_dropout_curves_faceted')
plt.show()

> **Interpretazione:**
> - Le curve di dropout rivelano **profili temporali diversi** tra i moduli. Alcuni corsi subiscono un'erosione costante e graduale; altri mostrano cali bruschi in punti specifici.
> - All'interno dello stesso modulo, presentazioni (coorti) diverse seguono traiettorie ampiamente simili, suggerendo che il pattern di dropout sia più una funzione della progettazione del corso che della composizione della coorte.
> - La maggior parte delle curve di dropout mostra un **segmento iniziale ripido** (abbandoni precoci) seguito da un declino più lento e graduale. Questo pattern è caratteristico delle curve di sopravvivenza in molti domini.
>
> **Parallelo SaaS:** Queste curve sono direttamente analoghe alle *curve di retention per coorte* nei business ad abbonamento. La forma della curva (convessa vs. concava, ripida vs. graduale) rivela se il churn è concentrato nell'onboarding (problema di attivazione) o distribuito nel tempo (problema di valore erogato).

## 4. Identificazione dei cliff di dropout

Non tutti i giorni sono uguali. Alcuni giorni registrano **numeri sproporzionatamente elevati di ritiri** — "dropout cliff" che possono corrispondere a eventi specifici del corso (scadenze delle valutazioni, pubblicazione dei voti, date limite per l'iscrizione).

Identifichiamo i cliff analizzando il conteggio giornaliero di dropout dai risultati della query BQ1 e segnalando i giorni in cui il conteggio supera il 95° percentile per quel corso. Sono i giorni in cui qualcosa ha innescato un numero insolito di abbandoni.

In [ ]:
# --- Cliff detection: days with unusually high dropout counts ---
# For each course-presentation, flag days where daily dropout count
# exceeds the 95th percentile. These outlier days represent sudden
# mass departures — likely tied to course milestones.
df_daily_dropouts = df_curves[[
    'code_module', 'code_presentation', 'dropout_day', 'n_dropouts'
]].copy()

# 95th percentile threshold per course-presentation
thresholds = (
    df_daily_dropouts
    .groupby(['code_module', 'code_presentation'])['n_dropouts']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'n_dropouts': 'p95_threshold'})
)

df_cliffs = df_daily_dropouts.merge(thresholds, on=['code_module', 'code_presentation'])
df_cliffs = df_cliffs[df_cliffs['n_dropouts'] >= df_cliffs['p95_threshold']]

# Aggregate across presentations: which module-day combos are cliffs most often?
df_cliff_summary = (
    df_cliffs
    .groupby(['code_module', 'dropout_day'])
    .agg(
        total_dropouts=('n_dropouts', 'sum'),
        n_presentations=('code_presentation', 'nunique'),
    )
    .reset_index()
    .sort_values('total_dropouts', ascending=False)
)

print('=== Top Cliff Events (days with >= p95 dropouts) ===\n')
print(df_cliff_summary.head(15).to_string(index=False))

# --- Visualization: top cliff events across all modules ---
df_cliff_top = df_cliff_summary.head(20)
fig, ax = plt.subplots(figsize=FIG_SIZE)
colors = [PALETTE_COURSE.get(m, '#999999') for m in df_cliff_top['code_module']]
ax.barh(
    range(len(df_cliff_top)),
    df_cliff_top['total_dropouts'],
    color=colors, edgecolor='white',
)

# Label each bar with module + day for identification
labels = [
    f"{row['code_module']} — day {int(row['dropout_day'])}"
    for _, row in df_cliff_top.iterrows()
]
ax.set_yticks(range(len(df_cliff_top)))
ax.set_yticklabels(labels)
ax.set_xlabel('Number of dropouts')
ax.set_title('Top Dropout Cliff Events (days exceeding 95th percentile)')
ax.invert_yaxis()
sns.despine()
fig.tight_layout()
save_fig(fig, '03_dropout_cliffs')
plt.show()

> **Interpretazione:**
> - Gli eventi cliff sono concentrati in **giorni specifici** che probabilmente corrispondono a milestone del corso: scadenze delle valutazioni, pubblicazione dei voti o date limite per l'iscrizione.
> - Alcuni moduli mostrano cliff più precoci nella timeline (suggerendo erosione nella fase di onboarding), mentre altri mostrano cliff a metà corso (suggerendo erosione guidata dalle valutazioni).
> - La coerenza dei tempi dei cliff tra presentazioni dello stesso modulo rafforza l'ipotesi che questi eventi siano legati alla **progettazione del corso** piuttosto che a fattori specifici dello studente.
>
> **Insight azionabile:** Se i giorni cliff coincidono con eventi noti del corso, l'operatore della piattaforma può implementare interventi mirati *prima* di quelle date — ad es. email di promemoria, risorse di supporto, o percorsi semplificati di re-iscrizione per gli studenti che hanno mancato le scadenze delle valutazioni.

## 5. Timeline normalizzata — Percentuale di avanzamento nel corso

I corsi OULAD hanno durate diverse (da ~240 a ~270 giorni). Confrontare i giorni di dropout in termini assoluti è come confrontare mele e arance. Per consentire il confronto tra corsi, `v_dropout_timing` include un campo `dropout_pct`: la percentuale di durata del corso trascorsa quando lo studente si è ritirato.

- 0% = ritirato all'inizio del corso (giorno 0)
- 50% = ritirato a metà corso
- 100% = ritirato alla fine programmata

I valori negativi rappresentano **ritiri pre-corso** (studenti che si sono disiscritti prima dell'inizio ufficiale del corso). Valori superiori al 100% sono possibili se uno studente si è ritirato dopo la data di fine programmata.

In [ ]:
# --- Normalized dropout timing distribution ---
# Using v_dropout_timing.dropout_pct to compare across courses.
# This DataFrame is reused in Section 6 (pre-course withdrawals).
df_dropout_timing = execute_query('SELECT * FROM v_dropout_timing')

# Separate in-course and pre-course withdrawals
df_in_course = df_dropout_timing[
    (df_dropout_timing['dropout_pct'] >= 0) & (df_dropout_timing['dropout_pct'] <= 100)
]
df_pre_course = df_dropout_timing[df_dropout_timing['dropout_pct'] < 0]

n_post = len(df_dropout_timing[df_dropout_timing['dropout_pct'] > 100])
print(f'Total withdrawals:        {len(df_dropout_timing):,}')
print(f'In-course (0-100%):       {len(df_in_course):,}')
print(f'Pre-course (< 0%):        {len(df_pre_course):,}')
print(f'Post-course (> 100%):     {n_post:,}')

# --- Histogram + KDE of normalized dropout timing ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

# Left: overall histogram for in-course withdrawals
ax1.hist(
    df_in_course['dropout_pct'], bins=50, color='#8172B3',
    edgecolor='white', alpha=0.8,
)
ax1.set_xlabel('Percentage through course at withdrawal')
ax1.set_ylabel('Number of withdrawals')
ax1.set_title('Dropout Timing Distribution (normalized)')
ax1.axvline(x=50, color='gray', linestyle='--', linewidth=1, label='Midpoint')
ax1.legend()
sns.despine(ax=ax1)

# Right: KDE by module for cross-course comparison
for module in sorted(df_in_course['code_module'].unique()):
    subset = df_in_course[df_in_course['code_module'] == module]
    # KDE requires enough data points to be meaningful
    if len(subset) > 10:
        sns.kdeplot(
            subset['dropout_pct'], ax=ax2,
            color=PALETTE_COURSE[module], label=module, linewidth=1.5,
        )
ax2.set_xlabel('Percentage through course at withdrawal')
ax2.set_ylabel('Density')
ax2.set_title('Dropout Timing by Module (KDE, normalized)')
ax2.legend(title='Module')
sns.despine(ax=ax2)

fig.suptitle('When Do Students Drop Out? (normalized by course length)',
             fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '03_dropout_normalized')
plt.show()

> **Interpretazione:**
> - L'istogramma normalizzato rivela le **fasi** del dropout: erosione precoce (0–20%), abbandono a metà corso (intorno al 30–50%) e ritiro in fase avanzata (oltre il 60%).
> - L'overlay KDE mostra che moduli diversi hanno **profili temporali di dropout diversi**. Alcuni corsi registrano la maggior parte del dropout nel primo quarto, mentre altri hanno un pattern più graduale o concentrato a metà corso.
> - Il fatto che una quota non trascurabile di ritiri avvenga oltre il 50% è notevole: questi studenti hanno investito tempo significativo prima di andarsene. Il dropout in fase avanzata potrebbe essere guidato dal fallimento nelle valutazioni o da difficoltà accademiche, piuttosto che dal disimpegno iniziale.
>
> **Parallelo SaaS:** La timeline normalizzata equivale all'*analisi per fase del ciclo di vita* nei business ad abbonamento. Il churn precoce (fallimento dell'attivazione) richiede interventi diversi rispetto al churn a metà ciclo (gap di valore) o al churn tardivo (spostamento competitivo o cambiamento di vita).

## 6. Ritiri pre-corso

Un sottoinsieme sorprendente di studenti si ritira **prima ancora che il corso inizi** (`dropout_day < 0`). Si tratta di iscrizioni in cui lo studente si è registrato e poi disiscritto prima del giorno 0.

I ritiri pre-corso rappresentano un fenomeno distinto: lo studente non ha mai sperimentato alcun contenuto del corso. Si tratta di puro **churn di registrazione** — analogo agli utenti SaaS che si iscrivono durante un trial e cancellano prima ancora di utilizzare il prodotto.

In [ ]:
# --- Pre-course withdrawal analysis ---
# Students who withdrew before day 0 (negative dropout_day).
df_precourse = execute_query('''
    SELECT
        code_module,
        COUNT(*) AS n_precourse,
        ROUND(
            100.0 * COUNT(*)
            / SUM(COUNT(*)) OVER (),
            1
        ) AS pct_of_precourse
    FROM v_dropout_timing
    WHERE dropout_day < 0
    GROUP BY code_module
    ORDER BY n_precourse DESC
''')

# What share of all withdrawals are pre-course?
total_withdrew_all = len(df_dropout_timing)
total_precourse = df_precourse['n_precourse'].sum()
precourse_pct = 100.0 * total_precourse / total_withdrew_all

print('=== Pre-Course Withdrawals ===\n')
print(f'  Total withdrawals:      {total_withdrew_all:>8,}')
print(f'  Pre-course (day < 0):   {total_precourse:>8,} ({precourse_pct:.1f}% of all withdrawals)')
print('\n=== Pre-Course by Module ===\n')
print(df_precourse.to_string(index=False))

# --- Bar chart: pre-course withdrawals by module ---
fig, ax = plt.subplots(figsize=FIG_SIZE)
colors = [PALETTE_COURSE.get(m, '#999999') for m in df_precourse['code_module']]
bars = ax.barh(
    df_precourse['code_module'], df_precourse['n_precourse'],
    color=colors, edgecolor='white',
)
for bar, (_, row) in zip(bars, df_precourse.iterrows()):
    ax.text(
        bar.get_width() + ax.get_xlim()[1] * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{int(row['n_precourse']):,} ({row['pct_of_precourse']:.1f}%)",
        va='center', fontsize=9, color='#333333',
    )
ax.set_xlabel('Number of pre-course withdrawals')
ax.set_title('Pre-Course Withdrawals by Module (withdrew before day 0)')
ax.invert_yaxis()
sns.despine()
fig.tight_layout()
save_fig(fig, '03_precourse_withdrawals')
plt.show()

> **Risultato chiave:** Una quota misurabile di tutti i ritiri avviene prima ancora che il corso inizi. Questi studenti si sono registrati ma non hanno mai interagito con alcun contenuto.
>
> - Il ritiro pre-corso è un segnale di **qualità della registrazione**: il funnel di iscrizione converte utenti che non sono ancora impegnati.
> - La distribuzione tra i moduli suggerisce che alcuni corsi potrebbero avere una conversione dalla registrazione all'avvio migliore di altri.
>
> **Opportunità di intervento:** I ritiri pre-corso potrebbero essere ridotti con una migliore impostazione delle aspettative al momento della registrazione, email di benvenuto con "primi passi" concreti, o un percorso semplificato di iscrizione tardiva. In termini SaaS, questo è il problema della "email di attivazione" — l'utente si è iscritto ma ha bisogno di una spinta per iniziare davvero.

## 7. Tempi di dropout per dati demografici

Il *momento* in cui gli studenti abbandonano varia in base al gruppo demografico? `v_dropout_timing` include colonne demografiche da `v_student_enriched`, permettendoci di segmentare i tempi di dropout per livello di istruzione, fascia d'età e indice di deprivazione.

**Distinzione importante:** Questa sezione è **solo descrittiva** — osserviamo differenze nel giorno mediano di dropout ma non eseguiamo test statistici. I test formali di ipotesi sulle associazioni demografiche appartengono al Notebook 05 (BQ3: dati demografici vs. comportamento).

In [ ]:
# --- Dropout timing by demographics ---
# Compute median dropout day for each demographic group.
# Using PERCENTILE_CONT for ANSI compliance.
# Only in-course withdrawals (dropout_day >= 0) to focus on the course experience.
df_demo_dropout = execute_query('''
    SELECT
        highest_education,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY dropout_day), 0)
            AS median_dropout_day,
        COUNT(*) AS n
    FROM v_dropout_timing
    WHERE dropout_day >= 0
    GROUP BY highest_education
    HAVING COUNT(*) >= 30
    ORDER BY median_dropout_day
''')

df_age_dropout = execute_query('''
    SELECT
        age_band,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY dropout_day), 0)
            AS median_dropout_day,
        COUNT(*) AS n
    FROM v_dropout_timing
    WHERE dropout_day >= 0
    GROUP BY age_band
    HAVING COUNT(*) >= 30
    ORDER BY median_dropout_day
''')

df_imd_dropout = execute_query('''
    SELECT
        COALESCE(imd_band, 'Unknown') AS imd_band,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY dropout_day), 0)
            AS median_dropout_day,
        COUNT(*) AS n
    FROM v_dropout_timing
    WHERE dropout_day >= 0
    GROUP BY 1
    HAVING COUNT(*) >= 30
    ORDER BY median_dropout_day
''')

# --- 1x3 bar chart panel ---
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

# Left: by education level
ax1.barh(df_demo_dropout['highest_education'], df_demo_dropout['median_dropout_day'],
         color='#4C72B0', edgecolor='white')
for i, (_, row) in enumerate(df_demo_dropout.iterrows()):
    ax1.text(
        row['median_dropout_day'] + 1, i,
        f"day {int(row['median_dropout_day'])} (n={int(row['n']):,})",
        va='center', fontsize=8, color='#333333',
    )
ax1.set_xlabel(LABEL_MEDIAN_DROPOUT_DAY)
ax1.set_title('By Education Level')
ax1.invert_yaxis()
sns.despine(ax=ax1)

# Center: by age band
ax2.barh(df_age_dropout['age_band'], df_age_dropout['median_dropout_day'],
         color='#55A868', edgecolor='white')
for i, (_, row) in enumerate(df_age_dropout.iterrows()):
    ax2.text(
        row['median_dropout_day'] + 1, i,
        f"day {int(row['median_dropout_day'])} (n={int(row['n']):,})",
        va='center', fontsize=8, color='#333333',
    )
ax2.set_xlabel(LABEL_MEDIAN_DROPOUT_DAY)
ax2.set_title('By Age Band')
ax2.invert_yaxis()
sns.despine(ax=ax2)

# Right: by IMD band
ax3.barh(df_imd_dropout['imd_band'], df_imd_dropout['median_dropout_day'],
         color='#C44E52', edgecolor='white')
for i, (_, row) in enumerate(df_imd_dropout.iterrows()):
    ax3.text(
        row['median_dropout_day'] + 1, i,
        f"day {int(row['median_dropout_day'])} (n={int(row['n']):,})",
        va='center', fontsize=8, color='#333333',
    )
ax3.set_xlabel(LABEL_MEDIAN_DROPOUT_DAY)
ax3.set_title('By IMD Band')
ax3.invert_yaxis()
sns.despine(ax=ax3)

fig.suptitle('Median Dropout Day by Demographic Group (in-course withdrawals only)',
             fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '03_dropout_by_demographics')
plt.show()

> **Interpretazione:**
> - Il giorno mediano di dropout varia tra i gruppi demografici, ma le differenze sono moderate rispetto alla variazione complessiva nei tempi di dropout.
> - Il **livello di istruzione** mostra una certa differenziazione: studenti con background educativi diversi potrebbero abbandonare in fasi diverse del ciclo di vita del corso.
> - La **fascia d'età** e la **fascia IMD** (indice di deprivazione socioeconomica) mostrano pattern che suggeriscono come il contesto demografico influenzi non solo *se* gli studenti abbandonano, ma *quando*.
>
> **Avvertenza:** Queste sono osservazioni descrittive — nessuna affermazione causale. Le differenze potrebbero essere confuse dalla scelta del corso, dall'esperienza pregressa o da altri fattori. BQ3 (Notebook 05) eseguirà test statistici formali confrontando predittori demografici e comportamentali.

## 8. Progettazione del corso e pattern di dropout

Le caratteristiche del corso spiegano la variazione nei tassi di dropout tra i moduli? `v_course_profile` fornisce metriche di progettazione per ogni corso-presentazione: durata, numero di valutazioni, densità delle valutazioni (valutazioni per 30 giorni), numero di risorse VLE e diversità dei tipi di attività.

Questa sezione cerca **correlazioni visive** tra le caratteristiche di progettazione del corso e il comportamento di dropout. Un confronto completo tra corsi è l'oggetto di BQ4 (Notebook 06); qui stabiliamo i pattern.

In [ ]:
# --- Course profile data ---
df_course = execute_query('SELECT * FROM v_course_profile')

# --- Median dropout day per course-presentation (in-course only) ---
df_course_dropout = execute_query('''
    SELECT
        code_module,
        code_presentation,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY dropout_day), 0)
            AS median_dropout_day
    FROM v_dropout_timing
    WHERE dropout_day >= 0
    GROUP BY code_module, code_presentation
''')

# Merge course profile with median dropout day
df_merged = df_course_dropout.merge(
    df_course[['code_module', 'code_presentation',
               'course_length_days', 'assessments_per_30_days',
               'withdrawal_rate_pct']],
    on=['code_module', 'code_presentation'],
)

# --- 1x2 scatter panel: course design vs dropout ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

# Left: assessment density vs withdrawal rate
for _, row in df_merged.iterrows():
    module = row['code_module']
    ax1.scatter(
        row['assessments_per_30_days'], row['withdrawal_rate_pct'],
        color=PALETTE_COURSE.get(module, '#999999'), s=80,
        edgecolor='white', zorder=3,
    )
    ax1.annotate(
        module,
        (row['assessments_per_30_days'], row['withdrawal_rate_pct']),
        fontsize=7, ha='center', va='bottom',
        xytext=(0, 5), textcoords='offset points',
    )
ax1.set_xlabel('Assessments per 30 days')
ax1.set_ylabel('Withdrawal rate (%)')
ax1.set_title('Assessment Density vs. Withdrawal Rate')
sns.despine(ax=ax1)

# Right: course length vs median dropout day
for _, row in df_merged.iterrows():
    module = row['code_module']
    ax2.scatter(
        row['course_length_days'], row['median_dropout_day'],
        color=PALETTE_COURSE.get(module, '#999999'), s=80,
        edgecolor='white', zorder=3,
    )
    ax2.annotate(
        module,
        (row['course_length_days'], row['median_dropout_day']),
        fontsize=7, ha='center', va='bottom',
        xytext=(0, 5), textcoords='offset points',
    )
# Reference line: dropout at course end (y = x diagonal)
max_len = df_merged['course_length_days'].max()
ax2.plot([0, max_len], [0, max_len], color='gray', linestyle='--',
         linewidth=1, alpha=0.5, label='Dropout at course end')
ax2.set_xlabel('Course length (days)')
ax2.set_ylabel(LABEL_MEDIAN_DROPOUT_DAY)
ax2.set_title('Course Length vs. Median Dropout Day')
ax2.legend(fontsize=8)
sns.despine(ax=ax2)

fig.suptitle('Course Design Characteristics and Dropout Patterns',
             fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '03_course_design_vs_dropout')
plt.show()

> **Interpretazione:**
> - **Densità delle valutazioni** e **tasso di ritiro** potrebbero mostrare una relazione: corsi con valutazioni più frequenti potrebbero avere profili di retention diversi. La direzione di questa relazione (se esiste) è una questione empirica che BQ4 esplorerà formalmente.
> - **Durata del corso** e **giorno mediano di dropout** mostrano che i corsi più lunghi tendono ad avere giorni mediani di dropout più tardivi in termini assoluti, il che è prevedibile — ma l'analisi normalizzata nella Sezione 5 ha già mostrato che il *timing relativo* (percentuale di avanzamento nel corso) potrebbe essere più coerente tra i moduli.
> - I pattern degli scatter plot suggeriscono che le caratteristiche di progettazione del corso contribuiscono — ma non spiegano completamente — la variazione nel comportamento di dropout. Anche i fattori a livello di studente (dati demografici, engagement) giocano un ruolo, come esploreranno BQ2 e BQ3.
>
> **Anteprima di BQ4:** Il Notebook 06 formalizzerà questa analisi con metriche di retention per corso, analisi dell'interazione con le valutazioni ed effetti della diversità delle risorse.

## 9. Conclusioni chiave e prossimi passi

### Cosa abbiamo imparato

1. **Il dropout è sostanziale e universale.** Circa un terzo di tutte le iscrizioni si conclude con un ritiro esplicito. Nessun modulo ne è immune, sebbene i tassi varino significativamente tra i corsi.

2. **Le curve cumulative di dropout rivelano profili temporali distinti.** Alcuni corsi subiscono un'erosione precoce ripida (fallimento dell'onboarding), mentre altri mostrano un declino più graduale a metà corso. La visualizzazione survival-style rende questi pattern immediatamente leggibili.

3. **Esistono eventi cliff.** Giorni specifici registrano numeri sproporzionatamente elevati di ritiri, probabilmente legati a milestone del corso (scadenze delle valutazioni, pubblicazione dei voti). Sono azionabili: gli interventi possono essere temporizzati prima delle date cliff note.

4. **Il timing normalizzato consente il confronto tra corsi.** Convertire il giorno di dropout in percentuale di avanzamento nel corso rivela che la *fase* del dropout (precoce, intermedia, tardiva) varia per modulo, suggerendo cause sottostanti diverse.

5. **I ritiri pre-corso sono un fenomeno distinto.** Una quota misurabile di studenti si ritira prima del giorno 0 — puro churn di registrazione. Questi studenti hanno bisogno di attivazione, non di supporto accademico.

6. **La segmentazione demografica mostra una variazione moderata.** I tempi di dropout differiscono leggermente per livello di istruzione, età e indice socioeconomico, ma le differenze sono modeste rispetto alla variazione complessiva. I test formali in BQ3 determineranno la significatività statistica.

7. **Le caratteristiche di progettazione del corso correlano con i pattern di dropout.** La densità delle valutazioni e la durata del corso mostrano associazioni visive con i tassi e i tempi di dropout, ma l'analisi formale in BQ4 è necessaria per quantificare queste relazioni.

### Cosa viene dopo

| Notebook | Business Question | Focus |
|----------|------------------|-------|
| **04** | BQ2 | Quali segnali comportamentali precoci predicono il dropout? |
| **05** | BQ3 | Dati demografici vs. comportamento — cosa predice meglio l'esito? |
| **06** | BQ4 | Come influiscono le caratteristiche del corso sulla retention? |
| **07** | BQ5 | Le 3 principali raccomandazioni operative |

---

**Riproducibilità:** Tutte le figure sono salvate in `reports/figures/`. Per riprodurre questo notebook, eseguire prima `python -m run_pipeline`, poi eseguire tutte le celle in ordine.

> **Dal timing ai segnali:** Questo notebook ha stabilito *quando* gli studenti se ne vanno — il panorama temporale del dropout. Ma il timing da solo non consente la prevenzione. Il passo successivo è identificare *quali comportamenti precoci predicono l'abbandono*.
>
> Proseguire con il **Notebook 04** (`04_bq2_early_signals.ipynb`) per BQ2: quali segnali comportamentali precoci predicono il dropout? Il Notebook 04 introduce i test statistici formali con t-test, effect size e correzione per confronti multipli — il primo utilizzo di `src/stats/tests.py` nell'analisi.